# Aegis Command - Time-Based Contiguous Holdout Fine-Tuning (Colab GPU Pipeline)

This notebook trains and evaluates the **Time-Based Contiguous Holdout Fine-Tuning Pipeline** (50 Epochs, Multi-scale 1280px) on Google Colab.

> **VERIFICATION PROTOCOL NOTICE**:
> Per project truth hierarchy, this notebook **ONLY dumps raw predictions and execution metadata** (`predictions.json` and `run_meta.json`).
> All AP50 deltas, 95% bootstrap confidence intervals, and comparison tables are **computed locally** on the workstation using `python3 scripts/compute_tables.py` and verified by `scripts/verify_benchmarks.py`.

### Temporal Leakage Prevention (Strict Contiguous Splits):
- **Fold 1**:
  - Train: Frames `0..107` (108 frames, 69.7%)
  - Temporal Purge Buffer: Frames `108..112` (5 frames purged to eliminate optical overlap)
  - Holdout Test: Frames `113..154` (42 frames, 27.1%)
- **Fold 2 (Reverse)**:
  - Holdout Test: Frames `0..41` (42 frames, 27.1%)
  - Temporal Purge Buffer: Frames `42..46` (5 frames purged)
  - Train: Frames `47..154` (108 frames, 69.7%)

### Google Drive Checkpointing:
- Trained weights are saved directly to Google Drive so progress is preserved across session disconnections.

In [ ]:
# 1. Verify GPU Allocation
!nvidia-smi

In [ ]:
# 2. Install Dependencies
%pip install -q ultralytics albumentations opencv-python matplotlib pandas numpy tabulate

In [ ]:
# 3. Clone Repository & Checkout Branch
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/accuracy-v2
!git pull

In [ ]:
# 4. Cryptographic Certification Gate Validation
from pathlib import Path
from vision.eval.eval_detection import check_verification_gate

marker = check_verification_gate(Path("vision/eval/labels"))
print(f"[GATE PASSED] Cryptographically verified ground truth at {marker}")

In [ ]:
# 5. CPU Pipeline Smoke Test (2-3 Images, 1 Epoch)
# [NOTE: Smoke test only to verify script execution without crashes. Not for accuracy evaluation.]
from vision.eval.split_dataset import split_dataset
from ultralytics import YOLO

print("Running CPU Smoke Test (1 epoch, tiny batch)...")
split_dataset(fold=1)

smoke_model = YOLO('yolov8n.pt')
smoke_results = smoke_model.train(
    data='vision/data/aegis_data.yaml',
    epochs=1,
    imgsz=320,
    batch=2,
    device='cpu',
    workers=0,
    project='runs/smoke',
    name='cpu_smoke_test',
    save=False,
    plots=False
)
print("[SMOKE TEST PASSED] Pipeline executed successfully end-to-end.")

In [ ]:
# 6. Mount Google Drive for Weight Checkpointing
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/aegis_checkpoints')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Google Drive mounted at {DRIVE_DIR}")

In [ ]:
# 7. Train Fold 1 (50 Epochs, GPU, yolov8s @ 1280px)
import shutil

split_dataset(fold=1)
print("Training Fold 1 (yolov8s backbone, 50 epochs, imgsz=1280)...\n")
model_f1 = YOLO('yolov8s.pt')
results_f1 = model_f1.train(
    data='vision/data/aegis_data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    box=7.5,
    cls=0.5,
    project='runs/train',
    name='aegis_fold1',
    save=True,
    plots=True
)

# Checkpoint best weights to Drive
f1_best = Path('runs/train/aegis_fold1/weights/best.pt')
if f1_best.exists():
    shutil.copy(f1_best, DRIVE_DIR / 'aegis_fold1_best.pt')
    print(f"Checkpointed Fold 1 weights to Drive: {DRIVE_DIR / 'aegis_fold1_best.pt'}")

In [ ]:
# 8. Train Fold 2 (50 Epochs, GPU, yolov8s @ 1280px)
split_dataset(fold=2)
print("Training Fold 2 (yolov8s backbone, 50 epochs, imgsz=1280)...\n")
model_f2 = YOLO('yolov8s.pt')
results_f2 = model_f2.train(
    data='vision/data/aegis_data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    box=7.5,
    cls=0.5,
    project='runs/train',
    name='aegis_fold2',
    save=True,
    plots=True
)

# Checkpoint best weights to Drive
f2_best = Path('runs/train/aegis_fold2/weights/best.pt')
if f2_best.exists():
    shutil.copy(f2_best, DRIVE_DIR / 'aegis_fold2_best.pt')
    print(f"Checkpointed Fold 2 weights to Drive: {DRIVE_DIR / 'aegis_fold2_best.pt'}")

In [ ]:
# 9. Raw Holdout Inference Dumps (Fold 1 & Fold 2, Baselines & Fine-Tuned)
from vision.eval.eval_detection import run_evaluation

# Fold 1: Pretrained Baseline on Val (Frames 113..154)
run_evaluation(
    model_path='yolov8s.pt',
    split='val',
    fold=1,
    imgsz=1280,
    device='cuda:0',
    native_classes=False,
    save_raw=True,
    raw_root='docs/results/raw',
    run_id='yolov8s_1280px_val_fold1_baseline'
)

# Fold 1: Fine-Tuned on Val (Frames 113..154)
run_evaluation(
    model_path='runs/train/aegis_fold1/weights/best.pt',
    split='val',
    fold=1,
    imgsz=1280,
    device='cuda:0',
    native_classes=True,
    save_raw=True,
    raw_root='docs/results/raw',
    run_id='yolov8s_1280px_val_fold1_finetuned'
)

# Fold 2: Pretrained Baseline on Val (Frames 0..41)
run_evaluation(
    model_path='yolov8s.pt',
    split='val',
    fold=2,
    imgsz=1280,
    device='cuda:0',
    native_classes=False,
    save_raw=True,
    raw_root='docs/results/raw',
    run_id='yolov8s_1280px_val_fold2_baseline'
)

# Fold 2: Fine-Tuned on Val (Frames 0..41)
run_evaluation(
    model_path='runs/train/aegis_fold2/weights/best.pt',
    split='val',
    fold=2,
    imgsz=1280,
    device='cuda:0',
    native_classes=True,
    save_raw=True,
    raw_root='docs/results/raw',
    run_id='yolov8s_1280px_val_fold2_finetuned'
)

print("\n[COMPLETE] All 4 holdout evaluation raw runs saved under docs/results/raw/")

In [ ]:
# 10. Package Raw Holdout Outputs and Weights for Download
!zip -r colab_train_raw.zip docs/results/raw/yolov8s_* runs/train/aegis_fold*/weights/best.pt

from google.colab import files
files.download('colab_train_raw.zip')
print("\n[DOWNLOAD READY] Extract colab_train_raw.zip into repo docs/results/raw/ and run scripts/compute_tables.py locally.")